# Day 22: Context Engineering and RAG
**What goes into the model's context, AI system architecture, and chunking documents**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai scikit-learn numpy pandas tiktoken

# ===== Option B: VS Code / local =====
# pip install openai scikit-learn numpy pandas tiktoken
# Create a .env file or set the variable in your terminal:  OPENAI_API_KEY=sk-...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
"""
import json, os, re
import numpy as np

def _get_key():
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")                 # change to any current chat model you have access to
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None):
    if ONLINE:
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}])
        return r.choices[0].message.content
    return offline_llm(prompt, system)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, ONLINE

## 2. The document: a sample employee handbook

In [ ]:
%%writefile handbook.md
# Sakura Logistics Employee Handbook (sample)

## 1. Working hours
Standard working hours are 9:30 to 18:30, Monday to Friday. Teams in the Tokyo office follow JST and teams in the Bengaluru office follow IST.
Flexible start times between 8:00 and 11:00 are allowed with manager approval.

## 2. Leave policy
Full-time employees receive 18 days of paid leave per year. Leave accrues monthly at 1.5 days.
Unused leave can be carried forward up to a maximum of 10 days. Carried forward leave expires on 31 March of the following year.
Sick leave is separate: employees receive 8 days of paid sick leave per year. A medical certificate is required for sick leave longer than 2 consecutive days.

## 3. Remote work
Employees may work remotely up to 2 days per week. Remote work must be logged in the HR portal by 10:00 on the same day.
Remote work from another country requires approval from HR and the security team at least 14 days in advance.

## 4. Expenses
Travel expenses must be submitted within 30 days with receipts. Meals during business travel are reimbursed up to 2,000 INR or 4,000 JPY per day.
Expenses above 50,000 INR require approval from a director before booking.

## 5. Security
Laptops must use full disk encryption. Customer data must never be copied to personal devices or uploaded to unapproved AI tools.
Security incidents must be reported to security@sakura-logistics.example within 1 hour of discovery.

In [ ]:
doc = open("handbook.md").read()
try:
    import tiktoken
    _enc = tiktoken.get_encoding("o200k_base")
    tokens = lambda t: len(_enc.encode(t))
except Exception:
    tokens = lambda t: max(1, round(len(t) / 4))
print(len(doc), "characters, about", tokens(doc), "tokens")

## 3. Three chunking strategies

In [ ]:
import re, numpy as np, pandas as pd

def fixed_chunks(text, size=300, overlap=50):
    return [text[i:i + size] for i in range(0, len(text), size - overlap)]

def paragraph_chunks(text):
    return [p.strip() for p in text.split("\n") if p.strip() and not p.startswith("#")]

def header_chunks(text):
    out = []
    for sec in re.split(r"\n(?=## )", text):
        lines = sec.strip().splitlines()
        if lines and lines[0].startswith("## "):
            title = lines[0][3:].strip()
            out.append({"section": title, "text": " ".join(lines[1:]).strip()})
    return out

strategies = {
    "fixed 300/50": fixed_chunks(doc),
    "paragraph": paragraph_chunks(doc),
    "headers": [c["text"] for c in header_chunks(doc)],
}
print(pd.DataFrame([{"strategy": k, "chunks": len(v), "avg_chars": int(np.mean([len(c) for c in v])),
                     "cut_mid_sentence": sum(not c.rstrip().endswith((".", ":")) for c in v)} for k, v in strategies.items()]).to_string(index=False))
print("\nExample of a broken fixed chunk:\n", [c for c in strategies["fixed 300/50"] if not c.rstrip().endswith(".")][1])

## 4. A boundary-aware chunker with contextual headers

In [ ]:
def smart_chunks(text, max_chars=350):
    chunks = []
    for sec in header_chunks(text):
        sentences = re.split(r"(?<=[.!?])\s+", sec["text"])
        cur = ""
        for s in sentences:
            if len(cur) + len(s) > max_chars and cur:
                chunks.append({"section": sec["section"], "text": cur.strip()})
                cur = ""
            cur += s + " "
        if cur.strip():
            chunks.append({"section": sec["section"], "text": cur.strip()})
    for i, c in enumerate(chunks, 1):
        c["id"] = f"c{i}"
        c["with_header"] = f"Employee Handbook > {c['section']}: {c['text']}"
    return chunks

chunks = smart_chunks(doc)
for c in chunks:
    print(c["id"], f"[{c['section']}]", c["text"][:80], "...")

## 5. Retrieval: with and without headers

In [ ]:
def retrieve(question, chunks, k=3, field="with_header"):
    vecs = embed([c[field] for c in chunks])
    q = embed(question)[0]
    sims = vecs @ q
    order = np.argsort(sims)[::-1][:k]
    return [(chunks[i], float(sims[i])) for i in order]

question = "When does carried forward leave expire?"
for field in ["text", "with_header"]:
    hits = retrieve(question, chunks, field=field)
    print(f"{field:<12}", [(h["id"], round(s, 3)) for h, s in hits])

## 6. Assemble context on a budget

In [ ]:
def assemble(parts, budget):
    kept, dropped, used = [], [], 0
    for name, text, prio in sorted(parts, key=lambda p: p[2]):
        t = tokens(text)
        if used + t <= budget:
            kept.append((name, text)); used += t
        else:
            dropped.append((name, t))
    return kept, used, dropped

rules = "You are an HR assistant for Sakura Logistics. Answer using ONLY the context. If the answer is not there, say \"I don't know\". Cite chunk ids like [c3]."
history = "Earlier the user asked about working hours and remote work. " * 20
docs = "\n".join(f"[{h['id']}] {h['text']}" for h, _ in retrieve(question, chunks))
kept, used, dropped = assemble([("rules", rules, 1), ("question", question, 1), ("docs", docs, 2), ("history", history, 3)], budget=250)
print("Kept:", [k for k, _ in kept], "| used", used, "tokens | dropped:", dropped)

## 7. End-to-end grounded answers

In [ ]:
def answer(question, k=3):
    hits = retrieve(question, chunks, k=k)
    context = "\n".join(f"[{h['id']}] {h['text']}" for h, _ in hits)
    prompt = f"Context:\n{context}\n\nQuestion: {question}"
    return llm(prompt, system=rules, temperature=0), [h["id"] for h, _ in hits]

for q in ["How many days of sick leave do employees get?",
          "When does carried forward leave expire?",
          "How far in advance must I request remote work from another country?",
          "Is there a company gym?"]:
    a, ids = answer(q)
    print(f"Q: {q}\nA: {a}\n   retrieved: {ids}\n")

### TODO 1: Change `max_chars` in `smart_chunks` to 150 and to 800. How do retrieval and answers change? Write two sentences on what you observe.

In [ ]:
# TODO: write your code here

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)